In [1]:
include("fdm/MHD_FDM.jl")
using .MHD_FDM, Random

include("spectral/MHDSpectral.jl")
using .MHDSpectral

using HDF5, Random

N::Int64 = 64
steps::Int = 200
truncate = 101
dt = 0.01

fω(x,y) = randn(Float64)
fj(x,y) = randn(Float64)

forcing(x,y) = sin.(x) + sin.(y)

ν = 2^-6
η = 2^-6

function data_post_process!(data::AbstractDict, savename::String)
    data["x"] = reshape(data["x"], 1, N)
    data["y"] = reshape(data["y"], 1, N)
    for key ∈ ("u", "v", "Bx", "By", "p")
        data[key] = data[key][:,:,truncate:end]
    end

    data["t"] = collect(0:(steps - truncate - 1))*dt

    fid = h5open(savename, "w")
    for (key, value) ∈ data
        fid["result/value/" * key * "/value"] = value
    end
    close(fid)
end

ibutcher_A = hcat(1)
ibutcher_b = [1]

data_spec = Dict()
data_fdm = Dict()

data_fdm["x"], data_fdm["y"], data_fdm["u"], data_fdm["v"], data_fdm["Bx"], data_fdm["By"], data_fdm["p"] = MHD_FDM.mhd_fdm(fω, fj, N, dt, ibutcher_A, ibutcher_b, steps, ν, η, false, true, "fdm/plots/", forcing)
data_spec["x"], data_spec["y"], data_spec["u"], data_spec["v"], data_spec["Bx"], data_spec["By"], data_spec["p"] = MHDSpectral.mhd_spectral(fω, fj, N, dt, ibutcher_A, ibutcher_b, steps, ν, η, false, true, "spectral/plots/", forcing)

data_post_process!(data_fdm, "data_rand_0.1_fdm.hdf5")
data_post_process!(data_spec, "data_rand_0.1_spec.hdf5")

LoadError: LoadError: SystemError: opening file "/home/viyengar/Documents/School/Research/SPIDER_Julia/src/sims/CommonSimTools.jl": No such file or directory
in expression starting at /home/viyengar/Documents/School/Research/SPIDER_Julia/src/sims/mhd/fdm/MHD_FDM.jl:1